Ниже — концептуальная архитектура пайплайна для предсказания **времени жизни (pot life) и скорости отверждения** полиуретановой системы.

---

### 1. Постановка задачи (Бизнес-логика)

**Проблема:** При замене катализатора или полиола в полиуретановой мастике меняется кинетика реакции. Чтобы найти новую идеальную пропорцию, химик делает 10–20 тестовых замесов. Это долго и дорого.
**Решение:** Обучить ML-модель на исторических данных лаборатории, чтобы она предсказывала время отверждения до того, как компоненты будут смешаны в реальности.

### 2. Подготовка данных (Feature Engineering)

Ваш опыт извлечения признаков из сырых временных рядов здесь будет как нельзя кстати. В химическом ML грамотно собранные фичи — это 80% успеха.

* **Таргет (Целевая переменная $y$):** Время достижения определенной вязкости или полного отверждения (в минутах).
* **Признаки (Features $X$):**
* Массовая доля NCO-групп (%).
* Концентрация катализатора (в ммоль/л).
* Начальная температура компонентов ($T_0$).
* Влажность воздуха (для влагоотверждаемых ПУ систем).
* **Синтетические фичи (Химические законы):** ML-модели работают лучше, если им «подсказать» физику процесса. Например, скорость реакции зависит от температуры нелинейно. Можно добавить фичу на базе уравнения Аррениуса:

$$k = A \exp\left(-\frac{E_a}{RT}\right)$$


### 3. Концептуальный код пайплайна на Python

Вот как может выглядеть ядро вашей системы. Это простой и понятный скрипт, который легко масштабировать.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# 1. Загрузка исторических данных лаборатории (симуляция)
# В реальности эти данные парсятся из Excel-журналов варок
data = {
    'temp_celsius': [20, 25, 22, 18, 30],
    'humidity_percent': [40, 50, 45, 60, 35],
    'nco_oh_ratio': [1.05, 1.10, 1.05, 1.15, 1.02],
    'catalyst_percent': [0.1, 0.15, 0.12, 0.2, 0.05],
    'target_cure_time_min': [120, 90, 105, 80, 150] # То, что мы предсказываем
}
df = pd.DataFrame(data)

# 2. Инженерия признаков (Feature Engineering)
# Добавляем нелинейную зависимость температуры (имитация физической кинетики)
df['temp_kelvin_inv'] = 1 / (df['temp_celsius'] + 273.15)

X = df.drop('target_cure_time_min', axis=1)
y = df['target_cure_time_min']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Сборка ML-пайплайна
# Используем Random Forest, так как он хорошо ловит нелинейные химические взаимодействия
pipeline = Pipeline([
    ('scaler', StandardScaler()), # Стандартизация признаков
    ('model', RandomForestRegressor(n_estimators=100, random_state=42))
])

# 4. Обучение модели
pipeline.fit(X_train, y_train)

# 5. Оценка точности (насколько мы ошибаемся в минутах)
predictions = pipeline.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
print(f"Средняя ошибка предсказания времени отверждения: {mae:.2f} минут")

# 6. ПРЕДСКАЗАНИЕ ДЛЯ НОВОГО ЭКСПЕРИМЕНТА (Бизнес-велью)
# Химик хочет узнать: "Что будет, если я положу 0.18% катализатора при 22°C и влажности 55%?"
new_experiment = pd.DataFrame({
    'temp_celsius': [22],
    'humidity_percent': [55],
    'nco_oh_ratio': [1.08],
    'catalyst_percent': [0.18],
    'temp_kelvin_inv': [1 / (22 + 273.15)]
})

predicted_time = pipeline.predict(new_experiment)
print(f"Прогнозируемое время отверждения новой рецептуры: {predicted_time[0]:.0f} минут")

Средняя ошибка предсказания времени отверждения: 16.70 минут
Прогнозируемое время отверждения новой рецептуры: 102 минут


Можно  описать этот пайплайн следующим образом:

1. **Сбор наследия:** «Сначала я оцифровываю логи предыдущих лабораторных испытаний и паспорта качества сырья».
2. **Обучение:** «Настраиваю ансамблевую модель (например, случайный лес или градиентный бустинг), которая находит скрытые нелинейные связи между дозировкой, температурой и временем жизни материала».
3. **Оптимизация:** «В итоге, когда нам нужно импортозаместить катализатор, мы не делаем 20 варок вслепую. Мы прогоняем параметры через модель, получаем 3-4 наиболее перспективные пропорции, и варим в лаборатории **только их**».

Такой подход сокращает время вывода нового продукта на рынок (Time-to-Market) в несколько раз. Для производственной компании, которая зависит от сезонности строительных работ, это убойный аргумент.